In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch



CAT = {
    "blue": "#2a78d6", "aqua": "#1baf7a", "yellow": "#eda100", "green": "#008300",
    "violet": "#4a3aa7", "red": "#e34948", "magenta": "#e87ba4", "orange": "#eb6834",
}

INK, INK_SECOND, INK_MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, SURFACE = "#e1e0d9", "#fcfcfb"
plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["DejaVu Sans", "Arial"],
    "text.color": INK, "axes.edgecolor": GRID, "axes.labelcolor": INK_SECOND,
    "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
    "axes.facecolor": SURFACE, "figure.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "figure.dpi": 120,
})
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


IE_MIN_COUNT = 5

BRANCH_MAP = {"Slavic": "Balto-Slavic", "Baltic": "Balto-Slavic",
              "Indo-Aryan": "Indo-Iranian", "Iranian": "Indo-Iranian",
              "Germanic": "Germanic", "Romance": "Romance", "Celtic": "Celtic"}
LEAF_COLOR = {"Germanic": CAT["red"], "Romance": CAT["orange"], "Celtic": CAT["green"],
              "Balto-Slavic": CAT["violet"], "Indo-Iranian": CAT["blue"]}
TARGET_BRANCHES = ["Germanic", "Romance", "Celtic", "Balto-Slavic", "Indo-Iranian"]

motif_long_ie = pd.read_pickle(f"/home/saba/folktale/analysis/output_agglo/motif_long_agglo.pkl")
with open(f"/home/saba/folktale/analysis/output_agglo/motif_cluster_info_agglo.json") as f:
    ie_info = json.load(f)
ie_labels = {int(k): v["label"] for k, v in ie_info.items()}

sub_ie = motif_long_ie[motif_long_ie["branch"].isin(BRANCH_MAP.keys())].copy()
sub_ie["branch2"] = sub_ie["branch"].map(BRANCH_MAP)
ct_ie = pd.crosstab(sub_ie["branch2"], sub_ie["cluster"]).reindex(index=TARGET_BRANCHES, fill_value=0)
char_sets_ie = {b: set(ct_ie.columns[ct_ie.loc[b] >= IE_MIN_COUNT]) for b in TARGET_BRANCHES}

n_countries_ie = motif_long_ie[motif_long_ie["branch"].isin(BRANCH_MAP.keys())].assign(
    branch2=lambda d: d["branch"].map(BRANCH_MAP)
).groupby("branch2")["culture"].nunique().to_dict()

ie_nodes = {b: dict(members=char_sets_ie[b], leaves=[b], is_leaf=True) for b in TARGET_BRANCHES}
ie_nodes["Proto-Italo-Celtic"] = dict(members=char_sets_ie["Romance"] & char_sets_ie["Celtic"],
                                        leaves=["Romance", "Celtic"], is_leaf=False)
ie_nodes["Proto-Germ-Italo-Celtic"] = dict(members=ie_nodes["Proto-Italo-Celtic"]["members"] & char_sets_ie["Germanic"],
                                             leaves=["Romance", "Celtic", "Germanic"], is_leaf=False)
ie_nodes["Proto-Western-IE"] = dict(members=ie_nodes["Proto-Germ-Italo-Celtic"]["members"] & char_sets_ie["Balto-Slavic"],
                                      leaves=["Romance", "Celtic", "Germanic", "Balto-Slavic"], is_leaf=False)
ie_nodes["Proto-Indo-European"] = dict(members=ie_nodes["Proto-Western-IE"]["members"] & char_sets_ie["Indo-Iranian"],
                                         leaves=TARGET_BRANCHES, is_leaf=False)


all_examples =  pd.read_pickle(f"/home/saba/folktale/analysis/output_agglo/motif_long_agglo_high_freq_motifs.pkl")
LEVEL_GAP = 2.6
ie_positions = {
    "Proto-Indo-European": (5.0, 0.0),
    "Proto-Western-IE": (3.0, -LEVEL_GAP),
    "Indo-Iranian": (8.8, -LEVEL_GAP),
    "Proto-Germ-Italo-Celtic": (1.6, -2 * LEVEL_GAP),
    "Balto-Slavic": (5.4, -2 * LEVEL_GAP),
    "Proto-Italo-Celtic": (0.2, -3 * LEVEL_GAP),
    "Germanic": (3.8, -3 * LEVEL_GAP),
    "Romance": (-1.2, -4 * LEVEL_GAP),
    "Celtic": (1.6, -4 * LEVEL_GAP),
}
ie_edges = [
    ("Proto-Indo-European", "Proto-Western-IE"), ("Proto-Indo-European", "Indo-Iranian"),
    ("Proto-Western-IE", "Proto-Germ-Italo-Celtic"), ("Proto-Western-IE", "Balto-Slavic"),
    ("Proto-Germ-Italo-Celtic", "Proto-Italo-Celtic"), ("Proto-Germ-Italo-Celtic", "Germanic"),
    ("Proto-Italo-Celtic", "Romance"), ("Proto-Italo-Celtic", "Celtic"),
]
BOX_W = 2.4
TALL_BOX_H, SHORT_BOX_H = 1.6, 0.85
HAS_EXAMPLES_IE = {"Proto-Indo-European", "Germanic", "Romance", "Celtic", "Balto-Slavic", "Indo-Iranian"}

fig, ax = plt.subplots(figsize=(15, 12))
for parent, child in ie_edges:
    px, py = ie_positions[parent]
    cx, cy = ie_positions[child]
    ph = TALL_BOX_H if parent in HAS_EXAMPLES_IE else SHORT_BOX_H
    ch = TALL_BOX_H if child in HAS_EXAMPLES_IE else SHORT_BOX_H
    my = (py - ph / 2 + cy + ch / 2) / 2
    ax.plot([px, px, cx, cx], [py - ph / 2, my, my, cy + ch / 2], color=INK_MUTED, linewidth=1.3, zorder=1)

for name, (x, y) in ie_positions.items():
    node = ie_nodes[name]
    is_leaf = node["is_leaf"]
    h = TALL_BOX_H if name in HAS_EXAMPLES_IE else SHORT_BOX_H
    color = LEAF_COLOR.get(name, INK_MUTED) if is_leaf else INK_SECOND
    box = FancyBboxPatch((x - BOX_W / 2, y - h / 2), BOX_W, h, boxstyle="round,pad=0.02,rounding_size=0.08",
                          linewidth=1.8 if is_leaf else 1.3, edgecolor=color, facecolor="white", zorder=2)
    ax.add_patch(box)
    n_members = len(node["members"])
    subtitle = f"[{n_members} shared motifs]" if not is_leaf else f"({n_members} motifs, {n_countries_ie.get(name, '?')} countries)"
    ax.text(x, y + h / 2 - 0.20, name, ha="center", va="top", fontsize=9.5, fontweight="bold",
            color=color if is_leaf else INK, zorder=3)
    ax.text(x, y + h / 2 - 0.44, subtitle, ha="center", va="top", fontsize=8, color=INK_SECOND, zorder=3)
    if name in HAS_EXAMPLES_IE:
        examples = all_examples.get(name, [])
        example_lines = [f"• {e}" for e in examples]
        if n_members > len(examples):
            example_lines.append(f"(+{n_members - len(examples)} more)")
        for i, ex in enumerate(example_lines):
            ax.text(x, y + h / 2 - 0.44 - 0.24 * (i + 1), ex, ha="center", va="top", fontsize=6.8, color=INK_MUTED, zorder=3)

ax.set_xlim(-2.6, 10.5)
ax.set_ylim(-4 * LEVEL_GAP - 1.2, 1.0)
ax.axis("off")

plt.tight_layout()
plt.show()
